# Experimento B — Traço Dominante, BFI-44, llama3.1:8b

Este notebook gera os agentes com o modelo `llama3.1:8b` e o instrumento **BFI-44** usando a GPU do Colab.

**Desenho**: cada agente tem 1 dos 5 traços sorteado como "dominante" (varia Baixo/Médio/Alto); os outros 4 ficam fixos em "Médio". Testa se induzir os 5 traços ao mesmo tempo (Experimento A) está diluindo/confundindo o sinal de cada traço individual, comparado a manipular só 1 por vez -- preocupação levantada pelo orientador. **Depende do Experimento A já ter rodado** (mesmo modelo e instrumento, só muda o desenho de indução).

## Pré-requisitos (faça antes de rodar)

1. **Ative a GPU:** `Ambiente de execução → Alterar tipo de ambiente de execução → GPU T4`. **Se a GPU não estiver disponível** (cota esgotada, alta demanda), não precisa esperar -- rode do mesmo jeito com ambiente de execução CPU (célula de verificação abaixo avisa, mas não trava). Fica mais lento, mas ainda roda melhor que localmente, e sem risco de faltar memória.
2. **Suba o projeto para o Google Drive:** copie a pasta `projeto-tcc` inteira para o seu Drive (ex: `Meu Drive/tcc/projeto-tcc/`). A pasta precisa ter:
   - `src/geracao/gerar_agentes.py` (já com suporte a `--modo multi`/`--modo dominante`)
   - `questionarios/perfis_personalidade.json`
   - `questionarios/bfi44.json`
   - `questionarios/ipa.json`

## Fluxo do notebook

1. Verifica GPU (avisa se não tiver, mas continua) → instala Ollama → baixa `llama3.1:8b`
2. Monta o Drive e copia os arquivos do projeto
3. Roda um **piloto de 30 agentes** para checar velocidade e qualidade
4. Analisa o piloto — decide se vale rodar os 300
5. Roda os **300 agentes completos** (dominante) e salva no Drive

### Sobre rodar em lotes pequenos (recomendado se a GPU cair)

Se a GPU do Colab não estiver disponível hoje, ou a sessão cair no meio, **não precisa recomeçar do zero**: o script grava cada agente no CSV assim que ele termina (não espera o lote todo). Basta rodar a célula "Retomar" (Passo 5) de novo sempre que conseguir abrir uma sessão nova -- ela conta quantos agentes já existem no arquivo e só gera o que falta. Dá pra fazer isso em vários lotes pequenos ao longo de vários dias, sem perder progresso.

---
## Passo 1 — Verificar GPU e instalar Ollama

In [ ]:
import subprocess

resultado = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True)
if resultado.returncode == 0:
    print('GPU disponivel:', resultado.stdout.strip())
else:
    print('ATENCAO: GPU nao detectada. Va em Ambiente de execucao > Alterar tipo > GPU T4')

In [ ]:
# zstd é necessário pelo instalador do Ollama a partir de versões recentes
!apt-get install -y -q zstd

# Instala o Ollama (detecta CUDA automaticamente)
!curl -fsSL https://ollama.ai/install.sh | sh

In [ ]:
import subprocess, time, os

# Após a instalação via shell, o PATH do kernel Python não é atualizado.
# O Ollama fica em /usr/local/bin/ollama — usamos o caminho completo.
OLLAMA_BIN = '/usr/local/bin/ollama'

if not os.path.exists(OLLAMA_BIN):
    raise FileNotFoundError(
        f'{OLLAMA_BIN} não encontrado.\n'
        'Verifique se a célula de instalação rodou sem erro e tente novamente.'
    )

print(f'Ollama encontrado em: {OLLAMA_BIN}')

# Inicia o servidor em background
subprocess.Popen([OLLAMA_BIN, 'serve'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                 start_new_session=True)
time.sleep(5)

# Confirma que está rodando
import requests
try:
    r = requests.get('http://localhost:11434')
    print('Ollama rodando:', r.text.strip())
except Exception as e:
    print('ERRO ao conectar no Ollama:', e)

In [ ]:
# Baixa o llama3.1:8b (~4.7 GB) — leva alguns minutos
!{OLLAMA_BIN} pull llama3.1:8b

In [ ]:
!{OLLAMA_BIN} list

---
## Passo 2 — Montar Google Drive e copiar arquivos do projeto

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil

# Ajuste este caminho se sua pasta estiver em outro lugar no Drive
DRIVE_PROJETO = '/content/drive/MyDrive/tcc/projeto-tcc'

# Pasta de trabalho local no Colab (mais rápida que Drive para I/O)
LOCAL_PROJETO = '/content/projeto-tcc'

if not os.path.exists(DRIVE_PROJETO):
    raise FileNotFoundError(
        f'Pasta nao encontrada: {DRIVE_PROJETO}\n'
        'Verifique se voce copiou a pasta projeto-tcc para o Drive '
        'e ajuste o caminho DRIVE_PROJETO acima.'
    )

# Copia o projeto para o Colab local (exceto resultados já existentes no Drive)
if os.path.exists(LOCAL_PROJETO):
    shutil.rmtree(LOCAL_PROJETO)
shutil.copytree(DRIVE_PROJETO, LOCAL_PROJETO)

# Garante que a pasta de resultados existe
os.makedirs(f'{LOCAL_PROJETO}/resultados', exist_ok=True)

print('Arquivos copiados para:', LOCAL_PROJETO)
print('Conteudo:')
for item in sorted(os.listdir(LOCAL_PROJETO)):
    print(' ', item)

---
## Passo 3 — Piloto: 30 agentes para checar velocidade e qualidade

Antes de rodar os 300 agentes, rodamos 30 para verificar:
- **Velocidade:** quantos segundos por agente na GPU?
- **Manipulação de personalidade:** correlação nível induzido × BFI-44
- **Variância do IPA:** % Conservador / Moderado / Arrojado

In [ ]:
import subprocess, os

# Garante que o PATH inclui o diretório do Ollama para o subprocesso
env = {**os.environ, 'AGENTE_MODEL': 'llama3.1:8b', 'PATH': f'/usr/local/bin:{os.environ.get("PATH", "")}'}

def garantir_ollama_rodando():
    """Confere se o servidor do Ollama está respondendo; se não estiver
    (ex.: o processo morreu depois de uma interrupção de célula),
    reinicia sozinho."""
    import requests, time
    try:
        if requests.get('http://localhost:11434', timeout=5).ok:
            return
    except Exception:
        pass
    print('Servidor do Ollama não está respondendo -- reiniciando...')
    subprocess.Popen([OLLAMA_BIN, 'serve'],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                     start_new_session=True)
    time.sleep(5)


def garantir_modelo(modelo='llama3.1:8b'):
    """Confere se o modelo já está disponível nesta sessão do Ollama; se
    não estiver (ex.: sessão do Colab reiniciou do zero e perdeu o modelo
    baixado antes), baixa de novo. Evita o erro 'HTTPError: 404 ... model
    not found' na hora de gerar."""
    r = subprocess.run([OLLAMA_BIN, 'list'], capture_output=True, text=True)
    if modelo not in r.stdout:
        print(f'Modelo {modelo} não encontrado nesta sessão -- baixando de novo (pode levar alguns minutos)...')
        pull = subprocess.run([OLLAMA_BIN, 'pull', modelo], capture_output=True, text=True)
        if pull.returncode != 0:
            print('ERRO ao baixar o modelo. Saída do Ollama:')
            print(pull.stdout[-2000:])
            print(pull.stderr[-2000:])
            raise RuntimeError(
                f"'ollama pull {modelo}' falhou (exit {pull.returncode}) -- "
                'veja a mensagem do Ollama acima para a causa real '
                '(rede, disco cheio, servidor caído, etc.).'
            )
        print('Download concluído.')
    else:
        print(f'Modelo {modelo} já disponível.')


garantir_ollama_rodando()
garantir_modelo('llama3.1:8b')

caminho_saida = f'{LOCAL_PROJETO}/resultados/piloto_exp_b_bfi44_dominante_8b.csv'
if os.path.exists(caminho_saida):
    with open(caminho_saida) as f:
        ja_existentes = sum(1 for _ in f) - 1
    if ja_existentes > 0:
        raise RuntimeError(
            f'{caminho_saida} já tem {ja_existentes} agente(s). Rodar '
            'esta célula de novo reaplicaria --seed do zero e DUPLICARIA os '
            'sorteios de personalidade dos agentes já existentes (mesmo bug '
            'que aconteceu no Experimento A em 2026-08-27). Apague o '
            'arquivo primeiro se quiser mesmo recomeçar o piloto do zero.'
        )

resultado = subprocess.run(
    ['python3', 'src/geracao/gerar_agentes.py', '--n', '30',
     '--output', 'resultados/piloto_exp_b_bfi44_dominante_8b.csv', '--modo', 'dominante', '--seed', '42'],
    cwd=LOCAL_PROJETO,
    env=env,
    capture_output=True, text=True
)
print(resultado.stdout)
if resultado.returncode != 0:
    print('ERRO:', resultado.stderr)

---
## Passo 4 — Análise do piloto

In [ ]:
import pandas as pd
from scipy import stats

CSV_PILOTO = f'{LOCAL_PROJETO}/resultados/piloto_exp_b_bfi44_dominante_8b.csv'
df = pd.read_csv(CSV_PILOTO)
df = df[df['ipa_perfil'].isin(['Conservador', 'Moderado', 'Arrojado'])]

DIMS = ['Extroversão', 'Afabilidade', 'Consciência', 'Neuroticismo', 'Abertura à Experiência']

print(f'Agentes válidos: {len(df)}')
print()

# Traço dominante de cada agente (só relevante no Experimento B)
if 'traco_dominante' in df.columns and df['traco_dominante'].notna().any():
    print('Distribuição de agentes por traço dominante sorteado:')
    print(df['traco_dominante'].value_counts().to_string())
    print()
# Velocidade
tempo_medio = (df['tempo_bfi44_s'] + df['tempo_ipa_s']).mean()
print(f'Velocidade média: {tempo_medio:.1f}s/agente')
print(f'Estimativa para 300 agentes: {tempo_medio * 300 / 60:.0f} minutos')
print()

# Distribuição do IPA
print('Distribuição do perfil IPA:')
print(df['ipa_perfil'].value_counts().to_string())
print()

# IPA score
q1_pts = df['ipa_q1_atribuida'].map({'a': 4, 'b': 3, 'c': 2, 'd': 1})
df['ipa_score_sem_q1'] = df['ipa_score'] - q1_pts
print(f'Escore IPA Q2-Q8: média={df["ipa_score_sem_q1"].mean():.2f}, '
      f'std={df["ipa_score_sem_q1"].std():.2f}, '
      f'min={df["ipa_score_sem_q1"].min()}, max={df["ipa_score_sem_q1"].max()}')
print()

# Checagem de manipulação
print('Checagem de manipulação (nível induzido × BFI-44):')
for d in DIMS:
    r, p = stats.spearmanr(df[f'nivel_{d}_num'], df[f'bfi_{d}'])
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
    print(f'  {d:<30}: r={r:+.3f} p={p:.3f} {sig}')
print()

# Correlação com IPA
print('Correlação (nível induzido × escore IPA sem Q1):')
for d in DIMS:
    r, p = stats.spearmanr(df[f'nivel_{d}_num'], df['ipa_score_sem_q1'])
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
    print(f'  {d:<30}: r={r:+.3f} p={p:.3f} {sig}')

---
## Passo 5 — Rodar os 300 agentes completos

**Execute esta célula apenas se o piloto mostrar resultados satisfatórios.**

O script usa modo append — se a sessão cair no meio, rode novamente com `--n <restante>` e ele continua de onde parou.

In [ ]:
OUTPUT_COMPLETO = 'resultados/principal/exp_b_bfi44_dominante_8b.csv'
env = {**os.environ, 'AGENTE_MODEL': 'llama3.1:8b', 'PATH': f'/usr/local/bin:{os.environ.get("PATH", "")}'}

def garantir_ollama_rodando():
    """Confere se o servidor do Ollama está respondendo; se não estiver
    (ex.: o processo morreu depois de uma interrupção de célula),
    reinicia sozinho."""
    import requests, time
    try:
        if requests.get('http://localhost:11434', timeout=5).ok:
            return
    except Exception:
        pass
    print('Servidor do Ollama não está respondendo -- reiniciando...')
    subprocess.Popen([OLLAMA_BIN, 'serve'],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                     start_new_session=True)
    time.sleep(5)


def garantir_modelo(modelo='llama3.1:8b'):
    """Confere se o modelo já está disponível nesta sessão do Ollama; se
    não estiver (ex.: sessão do Colab reiniciou do zero e perdeu o modelo
    baixado antes), baixa de novo. Evita o erro 'HTTPError: 404 ... model
    not found' na hora de gerar."""
    r = subprocess.run([OLLAMA_BIN, 'list'], capture_output=True, text=True)
    if modelo not in r.stdout:
        print(f'Modelo {modelo} não encontrado nesta sessão -- baixando de novo (pode levar alguns minutos)...')
        pull = subprocess.run([OLLAMA_BIN, 'pull', modelo], capture_output=True, text=True)
        if pull.returncode != 0:
            print('ERRO ao baixar o modelo. Saída do Ollama:')
            print(pull.stdout[-2000:])
            print(pull.stderr[-2000:])
            raise RuntimeError(
                f"'ollama pull {modelo}' falhou (exit {pull.returncode}) -- "
                'veja a mensagem do Ollama acima para a causa real '
                '(rede, disco cheio, servidor caído, etc.).'
            )
        print('Download concluído.')
    else:
        print(f'Modelo {modelo} já disponível.')


garantir_ollama_rodando()
garantir_modelo('llama3.1:8b')

caminho_saida = f'{LOCAL_PROJETO}/{OUTPUT_COMPLETO}'
if os.path.exists(caminho_saida):
    with open(caminho_saida) as f:
        ja_existentes = sum(1 for _ in f) - 1
    if ja_existentes > 0:
        raise RuntimeError(
            f'{caminho_saida} já tem {ja_existentes} agente(s). Rodar esta '
            'célula de novo reaplicaria --seed do zero e DUPLICARIA os '
            'sorteios de personalidade dos agentes já existentes (mesmo bug '
            'que aconteceu no Experimento A em 2026-08-27). Use a célula '
            '"Retomar" para continuar, ou apague o arquivo primeiro se '
            'quiser mesmo recomeçar do zero.'
        )

resultado = subprocess.run(
    ['python3', 'src/geracao/gerar_agentes.py', '--n', '300',
     '--output', OUTPUT_COMPLETO, '--modo', 'dominante', '--seed', '123'],
    cwd=LOCAL_PROJETO,
    env=env,
    capture_output=True, text=True
)
print(resultado.stdout[-3000:])  # últimas linhas do log
if resultado.returncode != 0:
    print('ERRO:', resultado.stderr[-1000:])

### Retomar após queda de sessão

Se a sessão do Colab cair no meio do run, use a célula abaixo para continuar de onde parou.

In [ ]:
import os, subprocess

OUTPUT_COMPLETO = 'resultados/principal/exp_b_bfi44_dominante_8b.csv'
caminho_csv = f'{LOCAL_PROJETO}/{OUTPUT_COMPLETO}'
env = {**os.environ, 'AGENTE_MODEL': 'llama3.1:8b', 'PATH': f'/usr/local/bin:{os.environ.get("PATH", "")}'}

def garantir_ollama_rodando():
    """Confere se o servidor do Ollama está respondendo; se não estiver
    (ex.: o processo morreu depois de uma interrupção de célula),
    reinicia sozinho."""
    import requests, time
    try:
        if requests.get('http://localhost:11434', timeout=5).ok:
            return
    except Exception:
        pass
    print('Servidor do Ollama não está respondendo -- reiniciando...')
    subprocess.Popen([OLLAMA_BIN, 'serve'],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                     start_new_session=True)
    time.sleep(5)


def garantir_modelo(modelo='llama3.1:8b'):
    """Confere se o modelo já está disponível nesta sessão do Ollama; se
    não estiver (ex.: sessão do Colab reiniciou do zero e perdeu o modelo
    baixado antes), baixa de novo. Evita o erro 'HTTPError: 404 ... model
    not found' na hora de gerar."""
    r = subprocess.run([OLLAMA_BIN, 'list'], capture_output=True, text=True)
    if modelo not in r.stdout:
        print(f'Modelo {modelo} não encontrado nesta sessão -- baixando de novo (pode levar alguns minutos)...')
        pull = subprocess.run([OLLAMA_BIN, 'pull', modelo], capture_output=True, text=True)
        if pull.returncode != 0:
            print('ERRO ao baixar o modelo. Saída do Ollama:')
            print(pull.stdout[-2000:])
            print(pull.stderr[-2000:])
            raise RuntimeError(
                f"'ollama pull {modelo}' falhou (exit {pull.returncode}) -- "
                'veja a mensagem do Ollama acima para a causa real '
                '(rede, disco cheio, servidor caído, etc.).'
            )
        print('Download concluído.')
    else:
        print(f'Modelo {modelo} já disponível.')


garantir_ollama_rodando()
garantir_modelo('llama3.1:8b')

if os.path.exists(caminho_csv):
    with open(caminho_csv) as f:
        ja_gerados = sum(1 for _ in f) - 1  # desconta cabeçalho
    restantes = 300 - ja_gerados
    print(f'Agentes já gerados: {ja_gerados} / 300')
    print(f'Faltam: {restantes}')

    if restantes > 0:
        resultado = subprocess.run(
            ['python3', 'src/geracao/gerar_agentes.py', '--n', str(restantes),
             '--output', OUTPUT_COMPLETO, '--modo', 'dominante'],
            cwd=LOCAL_PROJETO,
            env=env,
            capture_output=True, text=True
        )
        print(resultado.stdout[-3000:])
        if resultado.returncode != 0:
            print('ERRO:', resultado.stderr[-1000:])
    else:
        print('Run completo! Nada a retomar.')
else:
    print('Arquivo não encontrado. Rode o Passo 5 primeiro.')

---
## Passo 6 — Salvar resultados no Google Drive

In [ ]:
import shutil, os

# Pasta de destino no Drive
DRIVE_RESULTADOS = '/content/drive/MyDrive/tcc/projeto-tcc/resultados'
os.makedirs(DRIVE_RESULTADOS, exist_ok=True)

arquivos_para_salvar = [
    'resultados/piloto_exp_b_bfi44_dominante_8b.csv',
    'resultados/principal/exp_b_bfi44_dominante_8b.csv',
]

for arq in arquivos_para_salvar:
    origem = f'{LOCAL_PROJETO}/{arq}'
    destino = f'/content/drive/MyDrive/tcc/projeto-tcc/{arq}'
    if os.path.exists(origem):
        shutil.copy2(origem, destino)
        print(f'Salvo: {destino}')
    else:
        print(f'Nao encontrado (ainda nao gerado): {origem}')

---
## Passo 7 — Análise final dos 300 agentes

In [ ]:
import pandas as pd
import statsmodels.api as sm
from scipy import stats

df_full = pd.read_csv(f'{LOCAL_PROJETO}/resultados/principal/exp_b_bfi44_dominante_8b.csv')
df_full = df_full[df_full['ipa_perfil'].isin(['Conservador', 'Moderado', 'Arrojado'])]

DIMS = ['Extroversão', 'Afabilidade', 'Consciência', 'Neuroticismo', 'Abertura à Experiência']

q1_pts = df_full['ipa_q1_atribuida'].map({'a': 4, 'b': 3, 'c': 2, 'd': 1})
df_full['ipa_score_sem_q1'] = df_full['ipa_score'] - q1_pts

print(f'N total: {len(df_full)}')
print()
print('Distribuição de perfil IPA:')
print(df_full['ipa_perfil'].value_counts().to_string())
print()
print(f'Escore IPA Q2-Q8: média={df_full["ipa_score_sem_q1"].mean():.2f}, '
      f'std={df_full["ipa_score_sem_q1"].std():.2f}')
print()

print('Correlação de Spearman (nível induzido × IPA sem Q1) -- todos os agentes:')
for d in DIMS:
    r, p = stats.spearmanr(df_full[f'nivel_{d}_num'], df_full['ipa_score_sem_q1'])
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
    print(f'  {d:<30}: r={r:+.3f} p={p:.4f} {sig}')
print()

if 'traco_dominante' in df_full.columns and df_full['traco_dominante'].notna().any():
    print('Correlação (nível induzido × IPA), só entre os agentes onde aquele traço era o dominante:')
    for d in DIMS:
        sub = df_full[df_full['traco_dominante'] == d]
        if len(sub) >= 5:
            r, p = stats.spearmanr(sub[f'nivel_{d}_num'], sub['ipa_score_sem_q1'])
            sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
            print(f'  {d:<30}: N={len(sub):>3} r={r:+.3f} p={p:.4f} {sig}')
        else:
            print(f'  {d:<30}: N={len(sub)} (poucos agentes -- não calculado)')
    print()
print('AVISO: nesta OLS, para cada traço, ~80% dos agentes tem esse traço travado em Médio (só varia quando ele é o dominante do agente) -- a interpretação de \'controlando pelos demais\' não é a mesma da OLS do Experimento A. A comparação principal deste experimento é a correlação por traço dominante acima, não esta OLS (mantida só para referência).')
print('Regressão OLS (5 traços simultâneos → IPA sem Q1):')
X = sm.add_constant(df_full[[f'nivel_{d}_num' for d in DIMS]])
modelo = sm.OLS(df_full['ipa_score_sem_q1'], X).fit()
print(f'R²={modelo.rsquared:.4f}, F={modelo.fvalue:.3f}, p={modelo.f_pvalue:.4f}')
for d in DIMS:
    coef = modelo.params[f'nivel_{d}_num']
    p = modelo.pvalues[f'nivel_{d}_num']
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
    print(f'  {d:<30}: coef={coef:+.4f} p={p:.4f} {sig}')
print()

print('Checagem de manipulação (nível induzido × BFI-44):')
for d in DIMS:
    r, p = stats.spearmanr(df_full[f'nivel_{d}_num'], df_full[f'bfi_{d}'])
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
    print(f'  {d:<30}: r={r:+.3f} p={p:.4f} {sig}')